# Credit Card Fraud Detection

**Author:** Sebastian Tapia

Classification pipeline (Logistic Regression, Random Forest, Gradient Boosting, XGBoost) with SMOTE for class imbalance, hyperparameter tuning, cost-based threshold selection, and SHAP interpretability.

> **Note:** my professional background is in banks and insurance companies, environments that handle highly confidential information that cannot be shared or used outside those contexts. For that reason, this portfolio project is built entirely on a public dataset (Kaggle), with no data or information coming from my professional activity.

## Libraries used

In [ ]:
# Data handling and arrays.
import pandas as pd
import numpy as np

# Plots to understand the dataset's behavior.
import matplotlib.pyplot as plt
import seaborn as sns

# Splits data into train/test.
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score

# Standardizes features (mean 0, std 1)
from sklearn.preprocessing import StandardScaler

# Metrics used to evaluate the model beyond "accuracy".
from sklearn.metrics import (
                            classification_report,
                            confusion_matrix,
                            roc_auc_score,
                            precision_recall_curve,
                            roc_curve,
                            average_precision_score
                            )

# Chains steps together in an orderly way
from imblearn.pipeline import Pipeline

# Classification models used
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

# Creates synthetic examples of the minority class.
from imblearn.over_sampling import SMOTE

# Model interpretability
import shap

# To save the final model
import joblib

## Load the dataset

In [ ]:
# Load the CSV
df = pd.read_csv("creditcard.csv")

# Show the first rows to see the structure.
df.head()

# Data types, row count, nulls.
df.info()

# Basic statistics (mean, std, percentiles).
df.describe()

## Class distribution

In [ ]:
# Shows the proportion of fraud vs. non-fraud.
df["Class"].value_counts(normalize=True)

# Plots the distribution.
sns.countplot(data=df, x="Class")
plt.title("Class Distribution (Fraud vs. Non-Fraud)")
plt.show()

# Additional EDA — Amount and Time by class.
# With a 0.17% imbalance, the bar count doesn't say much more; what does
# help is seeing whether the amount or time of day behaves differently
# between fraud and non-fraud.

# Amount by class (log scale, since Amount has a very long tail)
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="Class", y="Amount")
plt.yscale("log")
plt.title("Amount Distribution by Class (Log Scale)")
plt.xlabel("Class (0 = Non-Fraud, 1 = Fraud)")
plt.ylabel("Amount (log)")
plt.show()

print("\nAmount statistics by class:")
print(df.groupby("Class")["Amount"].describe()[["mean", "50%", "max"]])

# Time by class — Time is measured in seconds since the first transaction
# in the dataset. It's converted to "hour of day" (the dataset spans ~2
# days) to make it interpretable. A temporary column is used (not added
# to df) so it doesn't leak into X/y further below.
df_eda = df.assign(Hour=(df["Time"] % (24 * 3600)) // 3600)

plt.figure(figsize=(8, 5))
sns.histplot(data=df_eda, x="Hour", hue="Class", stat="density", common_norm=False, bins=24)
plt.title("Transaction Distribution by Hour of Day, Normalized by Class")
plt.xlabel("Hour of day")
plt.ylabel("Density")
plt.show()

## Preprocessing

In [ ]:
X = df.drop("Class", axis=1)
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## Baseline model comparison (now with cross-validation, not a single split)

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        class_weight="balanced",
        random_state=42
    ),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

print("\n=== Baseline model comparison (CV, PR-AUC) ===\n")

for name, model in models.items():
    pipeline_cv = Pipeline([
        ("scaler", StandardScaler()),
        ("smote", SMOTE(random_state=42)),
        ("model", model)
    ])
    scores = cross_val_score(pipeline_cv, X_train, y_train, cv=cv, scoring="average_precision")
    print(f"{name}: PR-AUC CV = {scores.mean():.4f} ± {scores.std():.4f}")

## Ablation + automatic selection of the best model (includes XGBoost)

In [ ]:
# All candidate configs — RF under its balancing variants AND XGBoost —
# compete in the SAME ablation, with the SAME cross-validation. The real
# winner (best PR-AUC) is the one used as the final model, whichever it is.
print("\n=== Ablation: comparing all candidate configs (CV, PR-AUC) ===\n")

configs = {
    "RF - SMOTE only": Pipeline([
        ("scaler", StandardScaler()),
        ("smote", SMOTE(random_state=42)),
        ("model", RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42))
    ]),
    "RF - class_weight only": Pipeline([
        ("scaler", StandardScaler()),
        ("model", RandomForestClassifier(n_estimators=200, max_depth=10, class_weight="balanced", random_state=42))
    ]),
    "RF - SMOTE + class_weight": Pipeline([
        ("scaler", StandardScaler()),
        ("smote", SMOTE(random_state=42)),
        ("model", RandomForestClassifier(n_estimators=200, max_depth=10, class_weight="balanced", random_state=42))
    ]),
    "XGBoost - SMOTE": Pipeline([
        ("scaler", StandardScaler()),
        ("smote", SMOTE(random_state=42)),
        ("model", XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="logloss",
            random_state=42
        ))
    ]),
    "XGBoost - scale_pos_weight": Pipeline([
        ("scaler", StandardScaler()),
        ("model", XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="logloss",
            scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),  # XGB's equivalent of class_weight
            random_state=42
        ))
    ]),
}

ablation_results = {}
for name, pipe in configs.items():
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="average_precision")
    ablation_results[name] = scores.mean()
    print(f"{name}: {scores.mean():.4f} ± {scores.std():.4f}")

best_config = max(ablation_results, key=ablation_results.get)
print(f"\n>> Winning config: {best_config} (PR-AUC CV = {ablation_results[best_config]:.4f})")

## Hyperparameter search on the winning model

In [ ]:
# The ablation picks the balancing STRATEGY + algorithm; now we tune the
# hyperparameters of that specific config with RandomizedSearchCV, using
# the same CV so results stay comparable.
from sklearn.model_selection import RandomizedSearchCV
from sklearn.base import clone

print(f"\n=== Hyperparameter search for: {best_config} ===\n")

base_pipeline = clone(configs[best_config])
model_step = base_pipeline.named_steps["model"]

if isinstance(model_step, RandomForestClassifier):
    param_distributions = {
        "model__n_estimators": [100, 200, 300, 400],
        "model__max_depth": [4, 6, 8, 10, 12, None],
        "model__min_samples_leaf": [1, 2, 4, 8],
        "model__min_samples_split": [2, 5, 10],
    }
elif isinstance(model_step, XGBClassifier):
    param_distributions = {
        "model__n_estimators": [100, 200, 300, 400],
        "model__max_depth": [3, 4, 5, 6, 8],
        "model__learning_rate": [0.01, 0.05, 0.1, 0.2],
        "model__subsample": [0.6, 0.8, 1.0],
        "model__colsample_bytree": [0.6, 0.8, 1.0],
    }
else:
    param_distributions = {
        "model__C": [0.01, 0.1, 1.0, 10.0],
    }

search = RandomizedSearchCV(
    base_pipeline,
    param_distributions=param_distributions,
    n_iter=20,
    scoring="average_precision",
    cv=cv,
    random_state=42,
    n_jobs=-1,
)
search.fit(X_train, y_train)

print(f"Best hyperparameters: {search.best_params_}")
print(f"PR-AUC CV with tuning: {search.best_score_:.4f}  (before tuning: {ablation_results[best_config]:.4f})")


# ============================================================
# Final model training — uses the ablation's real winning config,
# now with tuned hyperparameters
# ============================================================

pipeline = search.best_estimator_  # already fit on X_train (refit=True is the default)


def evaluate_train_test(pipeline, X_train, y_train, X_test, y_test, name="Model"):
    """Train vs. test diagnostic (bias/variance)."""
    proba_train = pipeline.predict_proba(X_train)[:, 1]
    proba_test = pipeline.predict_proba(X_test)[:, 1]

    print(f"\n=== {name}: Train vs Test ===")
    print(f"Train  ROC-AUC: {roc_auc_score(y_train, proba_train):.4f} | PR-AUC: {average_precision_score(y_train, proba_train):.4f}")
    print(f"Test   ROC-AUC: {roc_auc_score(y_test, proba_test):.4f} | PR-AUC: {average_precision_score(y_test, proba_test):.4f}")
    return proba_test

## Initial evaluation

In [ ]:
y_proba = evaluate_train_test(pipeline, X_train, y_train, X_test, y_test, f"Final model ({best_config})")
y_pred = pipeline.predict(X_test)

print("\n=== Initial evaluation (test) ===\n")
print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

## Plots

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_proba)
plt.plot(fpr, tpr)
plt.title("ROC Curve")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.show()

precision, recall, thresholds = precision_recall_curve(y_test, y_proba)
plt.plot(recall, precision)
plt.title("Precision-Recall Curve")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.show()

importances = pipeline.named_steps["model"].feature_importances_
feature_names = X.columns

plt.figure(figsize=(10, 6))
sns.barplot(x=importances, y=feature_names)
plt.title(f"Feature Importance ({best_config})")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()

## Cost-based threshold selection

In [ ]:
false_positive_cost = 5     # cost of manually reviewing a legitimate transaction
false_negative_cost = 100   # cost of missing a real fraud case

threshold_results = []
for t in np.arange(0.05, 0.95, 0.05):
    pred = (y_proba >= t).astype(int)
    fp = ((pred == 1) & (y_test == 0)).sum()
    fn = ((pred == 0) & (y_test == 1)).sum()
    total_cost = fp * false_positive_cost + fn * false_negative_cost
    threshold_results.append((round(t, 2), total_cost))

optimal_threshold, min_cost = min(threshold_results, key=lambda x: x[1])
print(f"\nOptimal threshold: {optimal_threshold} | Estimated total cost: {min_cost}")
print("(adjust false_positive_cost / false_negative_cost to the real business case before reporting this)")

# Detailed report at the chosen threshold
threshold_pred = (y_proba >= optimal_threshold).astype(int)
print(confusion_matrix(y_test, threshold_pred))
print(classification_report(y_test, threshold_pred))


def score_transaction(row, model, threshold=optimal_threshold):
    # A one-row DataFrame is built with the same column names as X_train,
    # instead of a plain np.array — this way StandardScaler doesn't lose
    # the feature names and the UserWarning disappears.
    row_df = pd.DataFrame([row.values], columns=X_train.columns)
    p = model.predict_proba(row_df)[0][1]
    decision = "REVIEW" if p >= threshold else "APPROVE"
    return {"fraud_probability": p, "decision": decision}

## Production scoring examples

In [ ]:
print("\n=== Production scoring examples ===\n")
fraud_example = X_test[y_test == 1].iloc[0]
legit_example = X_test[y_test == 0].iloc[0]

print("Real fraud transaction:", score_transaction(fraud_example, pipeline))
print("Real legitimate transaction:", score_transaction(legit_example, pipeline))

## Interpretability

In [ ]:
explainer = shap.TreeExplainer(pipeline.named_steps["model"])
shap_values = explainer.shap_values(X_test)

shap.summary_plot(shap_values, X_test, plot_type="bar")

## Save the final model

In [ ]:
joblib.dump(pipeline, "fraud_model.pkl")
print("\nModel saved to fraud_model.pkl")

## Project summary (narrative wrap-up)

In [ ]:
print(f"""
=== Project summary ===
Model chosen: {best_config}
PR-AUC CV (before tuning): {ablation_results[best_config]:.4f}
PR-AUC CV (after tuning): {search.best_score_:.4f}
Best hyperparameters: {search.best_params_}
Threshold chosen: {optimal_threshold} (based on false_positive_cost={false_positive_cost}, false_negative_cost={false_negative_cost})
Most influential features (see SHAP plot above): fill in with the top 3-5
Business interpretation: at this threshold, X% of transactions are manually
reviewed and Y% of real fraud is caught (fill in with your numbers).
""")